# OPERA Path A -- two-stage pretrained long-context study (train @ 512, eval @ 8192)

Pre-registered protocol: `docs/OPERA_PathA_prereg.md` (read it first).
This notebook is a thin wrapper: it clones the repo, sets credentials,
and hands the session to `kaggle/patha_session.py`, which auto-runs the
next unfinished stage inside a wall-clock governor (clean stop +
checkpoint push before Kaggle's session cap).

**Kaggle session mechanics (read once):** changing the accelerator
(CPU <-> T4 x2) RESTARTS the session and WIPES /kaggle/working -- only
what was pushed to your Kaggle datasets (the engine does this after
every artifact and stage) or saved as a notebook version survives.
Datasets attached as inputs (Add Input) are notebook configuration:
they persist across restarts and remount read-only each session.
Run sessions with **Save & Run All** (Save Version), not interactive
Run All -- committed runs keep running server-side after you close the
tab AND persist /kaggle/working as a notebook version.

**One-time setup (Kaggle web UI, outside this notebook):**
1. Settings -> Accelerator: **none (CPU)** for the first (data)
   session, then **GPU T4 x2** after (never P100 -- confirmed broken).
2. Settings -> Internet -> **On**.
3. Add-ons -> Secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your
   kaggle.com Settings -> API -> Create New Token).
4. After the first session's push: Add Input -> your
   `opera-lm-patha-data` and `opera-lm-patha-ckpt` datasets.

**Each session:** Save & Run All, walk away.

In [ ]:
import os

# RE-RUN SAFETY: a previous run of this cell may have %cd'd into
# /kaggle/working/repo, which the rm -rf below deletes -- every
# !command then dies with "shell-init: error retrieving current
# directory". Restore a valid CWD in Python BEFORE any shell command.
os.chdir("/kaggle/working")

REPO_URL = "https://github.com/Merna-Khalid/OPERA-LM"
COMMIT = "main"        # or pin a hash for exact reproducibility

!rm -rf /kaggle/working/repo
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout -q $COMMIT
!pip install -q datasets tokenizers huggingface_hub kaggle

from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = sec.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = sec.get_secret("KAGGLE_KEY")
os.environ.setdefault("GOVERNOR_HOURS", "10.5")

assert os.path.isdir("/kaggle/working/repo/kaggle"), \
    "clone failed -- is Internet On in Settings?"
print("setup ok:", os.getcwd(), "| commit:", COMMIT,
      "| governor hours:", os.environ["GOVERNOR_HOURS"])

## Session engine

`--stages auto` picks: env -> data (first, CPU-only session is fine) ->
smoke -> pretrain_opera -> pretrain_tf (pair) -> sft_opera -> sft_tf ->
curves. Force a stage with e.g. `--stages data`; inspect with
`--stages status`. Output streams live; full logs also land in
`/kaggle/working/patha/logs/`.

In [ ]:
!python /kaggle/working/repo/kaggle/patha_session.py --stages auto

In [ ]:
!python /kaggle/working/repo/kaggle/patha_session.py --stages status